In [ ]:
# import 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt
import mplotutils as mpu
from utils import process_data


from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/

#activate interactive figures
#%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

#directory for figure saving
#dir_save = 

# Save figure or not
save_fig = True

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import mplotutils as mpu
from cartopy.feature import ShapelyFeature
from cartopy.io.shapereader import Reader,natural_earth

def plot_towns(
    ax, lats, lons, resolution="10m", transform=ccrs.PlateCarree(), zorder=3
):
    """
    This function will download the 'populated_places' shapefile from
    NaturalEarth, trim the shapefile based on the limits of the provided
    lat & long coords, and then plot the locations and names of the towns
    on a given GeoAxes.

    ax = a pyplot axes object
    lats = latitudes
    lons = longitudes
    resolution= str. either high res:'10m' or low res: '50m'
    transform = a cartopy crs object

    From https://gist.github.com/cbur24/9122c763301f6de1d6e1fc39763c2444
    """
    # get town locations
    shp_fn = natural_earth(
        resolution=resolution, category="cultural", name="populated_places"
    )
    shp = Reader(shp_fn)
    xy = [pt.coords[0] for pt in shp.geometries()]
    x, y = list(zip(*xy))

    # get town names
    towns = shp.records()
    names_en = []
    for town in towns:
        names = town.attributes["NAME"]
        names_en.append(names)

    # create data frame and index by the region of the plot
    all_towns = pd.DataFrame({"names_en": names_en, "x": x, "y": y})
    region_towns = all_towns[
        (all_towns.y < np.max(lats))
        & (all_towns.y > np.min(lats))
        & (all_towns.x > np.min(lons))
        & (all_towns.x < np.max(lons))
    ]

    # plot the locations and labels of the towns in the region
    ax.scatter(
        region_towns.x.values,
        region_towns.y.values,
        c="black",
        marker=".",
        transform=transform,
        zorder=zorder,
    )
    transform_mpl = ccrs.PlateCarree()._as_mpl_transform(
        ax
    )  # this is a work-around to transform xy coords in ax.annotate
    for i, txt in enumerate(region_towns.names_en):
        ax.annotate(
            txt,
            (region_towns.x.values[i], region_towns.y.values[i]),
            xycoords=transform_mpl,
            textcoords="offset points",
            xytext=(3, 3), #text offset in points
        )

In [ ]:

[lon1, lon2, lat1, lat2] = [33, 43, -5, 6]
projection = ccrs.PlateCarree()  # ccrs.Orthographic(central_latitude=40)
fig, ax = plt.subplots(
    subplot_kw=dict(projection=projection)
)  # , figsize=(8, 8))
ax.set_extent([lon1, lon2, lat1, lat2], crs=projection)

# lands = cfeature.NaturalEarthFeature(category='physical', name='land',scale= '50m',
#                                     edgecolor='black',facecolor=None) #edgecolor=frontcol,facecolor=landcol
#countries = cfeature.NaturalEarthFeature('cultural','admin_0_countries','50m')
#ax.add_feature(countries, linestyle=":", edgecolor="black",facecolor=None)
#ax.add_feature(lands, zorder=0, facecolor=None)
# ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="black",scale= '50m',)

# lakes = cfeature.GSHHSFeature(scale='auto', levels=[2]) # GSHHSF: level 1 = coastlines, level2 = lakes
# ax.add_feature(lakes)
# # ax.add_feature(cfeature.LAKES)
#ax.add_feature(cfeature.RIVERS,scale= '50m')
# # Add Kenya county borders
filename_counties = r".\plotting\kenyan-counties\County.shp"
counties = ShapelyFeature(Reader(filename_counties).geometries(), ccrs.PlateCarree(), 
                            linewidth = 1, 
                            facecolor = 'lightgrey', 
                            edgecolor = 'black',
                            linestyle=':'
                            )
ax.add_feature(counties)
# filename_towns = r".\plotting\ken_major_towns\ken_major_towns.shp"
# towns = ShapelyFeature(Reader(filename_towns).geometries(), ccrs.PlateCarree(), 
#                             linewidth = 1, 
#                             facecolor = 'red', 
#                             edgecolor = 'black',
#                             linestyle='-',
#                             )
# ax.add_feature(towns)

# add towns manually:
#points = list(Reader(filename_towns).geometries())
# ax.scatter([point.x for point in points],
#            [point.y for point in points],
#            transform=ccrs.PlateCarree())

#plot town locations from global dataset:
lons = np.arange(lon1,lon2,0.1)
lats = np.arange(lat1,lat2,0.1)
plot_towns(ax,lats,lons)

# set the ticks
lons = np.arange(lon1,lon2, 2)
lats = np.arange(lat1,lat2,1)
ax.set_xticks(lons, crs=ccrs.PlateCarree())
ax.set_yticks(lats, crs=ccrs.PlateCarree())

# format the ticks as e.g 60°W
ax.xaxis.set_major_formatter(LongitudeFormatter())
ax.yaxis.set_major_formatter(LatitudeFormatter())


# ax.set_ylabel("latitude")
# ax.set_xlabel("lon")
ax.plot(mknlon, mknlat, marker="o", color="red")  # mt. kenya statio
plt.show()